# 02 – Mehrszenarienvergleich gespeicherter Zustände

Liest ausschließlich gespeicherte Szenarioergebnisse aus `results/szenarien/<ID>/` (Kennwerte.csv, Verteilung.csv, laufprotokoll.json) mit Integritätsprüfung.
**Keine Zustandsbildung, keine Energieberechnung, keine Flussrekonstruktion.** Export nur mit `EXPORTIEREN=True`; ein vorhandener Vergleich wird nur mit `ERSETZEN=True` ersetzt.

Ablauf: Szenarien auswählen → gemeinsame Kennwerttabelle laden → Vergleichbarkeit prüfen → Gesamtübersicht → Referenzvergleich → optional Paarvergleich → optional Export.

Die Rechenlogik liegt in `modell/szenarienvergleich.py`; das Notebook ruft nur dessen öffentliche Funktionen auf.

## Start

- **Was:** Projektordner finden, Modell- und Vergleichsmodul laden.
- **Eingaben:** `config/projekt.json`
- **Rechnet oder liest:** liest nur
- **Annahmen:** –
- **Interpretation:** –
- **Sperrende Fehler:** Projektordner nicht gefunden.

In [3]:
# Einheitliche Startzelle: Projektordner suchen, aktuelle Modellmodule laden, Fassung prüfen, Projektpfade lesen.
import os, sys
from pathlib import Path
try:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'config' / 'projekt.json').is_file())
except StopIteration:
    raise FileNotFoundError('Projektordner nicht gefunden: Das Notebook muss im Ordner notebooks/ des Projekts liegen (config/projekt.json fehlt).') from None
MODELL = ROOT / 'modell'
if str(MODELL) not in sys.path:
    sys.path.insert(0, str(MODELL))
for _name in [n for n, m in list(sys.modules.items()) if os.path.normcase(str(getattr(m, '__file__', None) or '')).startswith(os.path.normcase(str(MODELL)))]:
    del sys.modules[_name]   # nach einer Aktualisierung immer die Module aus dem Projektordner laden (kein veralteter Kernelstand)
import fassung
print('Projektordner:', ROOT)
print(fassung.pruefe_notebook('02_Szenarienvergleich', '1.6.3-AP6'))
import gebaeudesektor as gs
from IPython.display import display
PROJEKT = gs.lade_projekt(ROOT)
import szenarienvergleich as sv

Projektordner: c:\Users\ma0026.NB-MA0026\Documents\peq-higrube\Berechnungen\gebaeudesektormodell\Gebaeudesektor_konsolidiert\Gebaeudesektor
Fassung 1.6.3-AP6 (Notebook und Modell passen zusammen)


## Verfügbare Vergleiche und Ergebnisse

- **Was:** Registrierte Vergleiche und gespeicherte Szenarioergebnisse anzeigen.
- **Eingaben:** `config/vergleiche.json`, `results/szenarien/`
- **Rechnet oder liest:** liest nur
- **Annahmen:** –
- **Interpretation:** Nur Szenarien mit Status OK und vorhandener `Kennwerte.csv` (Format ab AP1) können verglichen werden.
- **Sperrende Fehler:** ungültiges Vergleichsregister

In [4]:
display(sv.registrierte_vergleiche(PROJEKT))
display(gs.ergebnisuebersicht(PROJEKT))

,Vergleich_ID,Szenarien,Referenz,Paar,Beschreibung
0,Bestand_2024__Transition_2040,"Bestand_2024, Transition_2040",Bestand_2024,Bestand_2024 → Transition_2040,Ausgangsbestand 2024 gegen Transitionszustand ...
1,Bestand_Transition_Wachstum_2040,"Bestand_2024, Transition_2040, Transition_2040...",Bestand_2024,Transition_2040 → Transition_2040_Wachstum,Bestand 2024 gegen zwei Zielbilder 2040: konst...
2,Bestand_Transition_ambitioniert_2040,"Bestand_2024, Transition_2040_ambitioniert, Tr...",Bestand_2024,Transition_2040_ambitioniert → Transition_2040...,Bestand 2024 gegen ambitioniertes 2040


,Szenario_ID,Ordner,Status,Jahr,NGF_m2,Positive_Typen,Zustand_ID,Energie,Kennwerte,Integritaet
0,Bestand_2024,results/szenarien/Bestand_2024,LESBAR,2024,889872558.0,14640,6d9c1ee883204a04,Ausgeführt,True,OK (SHA-256 der Ausgaben geprüft)
1,Transition_2040,results/szenarien/Transition_2040,LESBAR,2040,889872558.0,16320,e07bff047ea3de97,Ausgeführt,True,OK (SHA-256 der Ausgaben geprüft)
2,Transition_2040_ambitioniert,results/szenarien/Transition_2040_ambitioniert,LESBAR,2040,950000000.0,8160,7efe32919bdba57d,Ausgeführt,True,OK (SHA-256 der Ausgaben geprüft)
3,Transition_2040_Wachstum,results/szenarien/Transition_2040_Wachstum,LESBAR,2040,950000000.0,15336,cbf1cf4d08e5df11,Ausgeführt,True,OK (SHA-256 der Ausgaben geprüft)


## Steuerung

- **Was:** Auswahl des Vergleichs; optional freie Auswahl von Szenarien, Referenz und Paar.
- **Eingaben:** Vergleichs-ID aus `config/vergleiche.json` oder eigene Liste
- **Rechnet oder liest:** –
- **Annahmen:** Schalter standardmäßig `False`. Ohne eigene Auswahl gilt der registrierte Vergleich.
- **Interpretation:** –
- **Sperrende Fehler:** –

In [5]:
VERGLEICH = 'Bestand_Transition_ambitioniert_2040'   # registrierter Vergleich bzw. Name des Exportordners
SZENARIEN = None      # z. B. ['Bestand_2024', 'Transition_2040', 'Szenario_X'] – überschreibt die Registrierung
REFERENZ = None       # None: Referenz aus der Registrierung bzw. erstes Szenario
PAAR = ["Bestand_2024", "Transition_2040_ambitioniert"]           # z. B. ['Bestand_2024', 'Transition_2040']; None: Paar aus der Registrierung (bei eigener Auswahl: kein Paar)
MINDESTABDECKUNG_PCT = None   # None: Wert aus der Registrierung (Standard 95 %)
EXPORTIEREN = True   # Vergleich nach results/vergleiche/<VERGLEICH>/ speichern
ERSETZEN = True      # vorhandenen Vergleich ersetzen (nur bewusst setzen)

## Gemeinsame Kennwerttabelle laden und Vergleichbarkeit prüfen

- **Was:** Lädt die Kennwerttabellen aller gewählten Szenarien und bewertet jeden Kennwert gegenüber der Referenz.
- **Eingaben:** gespeicherte Szenarioergebnisse
- **Rechnet oder liest:** liest; Vergleichbarkeit und Differenzen werden aus gespeicherten Kennwerten gebildet
- **Annahmen:** Vergleichbar nur bei gleicher Einheit, Bezugsfläche und Vergleichsbasis; Abdeckung ≥ Mindestabdeckung (sonst NICHT_VERGLEICHBAR), unter 100 % BEDINGT_VERGLEICHBAR.
- **Interpretation:** Nicht vergleichbare oder fehlende Energiewerte sperren den Flächenvergleich nicht.
- **Sperrende Fehler:** Ergebnis fehlt, Integritätsprüfung fehlgeschlagen, Kennwerte.csv fehlt (Altformat)

In [6]:
if SZENARIEN is None:
    auswahl = sv.vergleichsauswahl(VERGLEICH, PROJEKT)
else:
    auswahl = {'szenarien': SZENARIEN, 'referenz': REFERENZ, 'paar': PAAR, 'mindestabdeckung_pct': sv.STANDARD_MINDESTABDECKUNG_PCT}
if REFERENZ is not None:
    auswahl['referenz'] = REFERENZ
if PAAR is not None:
    auswahl['paar'] = PAAR
if MINDESTABDECKUNG_PCT is not None:
    auswahl['mindestabdeckung_pct'] = MINDESTABDECKUNG_PCT
erg = sv.vergleiche(auswahl['szenarien'], auswahl['referenz'], auswahl['paar'], PROJEKT, VERGLEICH, auswahl['mindestabdeckung_pct'])
print('Referenz:', erg['referenz'], '| Paar:', erg['paar'], '| Mindestabdeckung:', erg['mindestabdeckung_pct'], '%')
display(erg['meta'])
display(sv.statusuebersicht(erg))

Referenz: Bestand_2024 | Paar: ['Bestand_2024', 'Transition_2040_ambitioniert'] | Mindestabdeckung: 95.0 %


,Szenario_ID,Jahr,Zustand_ID,NGF_m2,Positive_Typen,Energie,Flaechenbasis,Energiebasis,Integritaet,Ordner
0,Bestand_2024,2024,6d9c1ee883204a042747bf69783d52e4d51ced7eafdacd...,889872558.0,14640,Ausgeführt,F-507e7b1fc41dcd82,E-871aa7fd993a65cc,OK (SHA-256 der Ausgaben geprüft),results/szenarien/Bestand_2024
1,Transition_2040_ambitioniert,2040,7efe32919bdba57d4c0e4f085aa6e06296d2bceac9ce2c...,950000000.0,8160,Ausgeführt,F-507e7b1fc41dcd82,E-871aa7fd993a65cc,OK (SHA-256 der Ausgaben geprüft),results/szenarien/Transition_2040_ambitioniert
2,Transition_2040_Wachstum,2040,cbf1cf4d08e5df11502ae786b95f1f4f16f443ad5992bb...,950000000.0,15336,Ausgeführt,F-507e7b1fc41dcd82,E-871aa7fd993a65cc,OK (SHA-256 der Ausgaben geprüft),results/szenarien/Transition_2040_Wachstum


,Referenz,Szenario_ID,Art,VERGLEICHBAR,BEDINGT_VERGLEICHBAR,NICHT_VERGLEICHBAR,KENNWERT_FEHLT
0,Bestand_2024,Transition_2040_Wachstum,Energie,87,0,0,0
1,Bestand_2024,Transition_2040_Wachstum,Fläche,126,0,0,0
2,Bestand_2024,Transition_2040_ambitioniert,Energie,87,0,0,0
3,Bestand_2024,Transition_2040_ambitioniert,Fläche,126,0,0,0


## Gesamtübersicht

- **Was:** Alle Szenarien nebeneinander: Flächen, Anteile und Energiekennwerte.
- **Eingaben:** gemeinsame Kennwerttabelle
- **Rechnet oder liest:** liest
- **Annahmen:** –
- **Interpretation:** Zustandswerte je Jahr; keine Aussagen über Flüsse zwischen den Zuständen.
- **Sperrende Fehler:** –

In [7]:
display(erg['uebersicht'])
sv.grafik_uebersicht(erg).show()

Szenario_ID,Bereich,Gruppe,Kennwert,Kategorie,Einheit,Bestand_2024,Transition_2040_ambitioniert,Transition_2040_Wachstum
0,Gesamt,Fläche,NGF,,m²,8.898726e+08,9.500000e+08,9.500000e+08
1,Gesamt,Fläche,BGF,,m²,1.112341e+09,1.187500e+09,1.187500e+09
2,Gesamt,Typen,Positiv belegte Typen,,Anzahl,1.464000e+04,8.160000e+03,1.533600e+04
3,Haushalte,Fläche,NGF,,m²,5.656768e+08,6.038988e+08,6.038988e+08
4,Haushalte,Fläche,BGF,,m²,7.070960e+08,7.548735e+08,7.548735e+08
...,...,...,...,...,...,...,...,...
208,Nichtwohnen,Energie: Endenergieträger,Endenergie,Erdgas,GWh/a,1.252825e+04,0.000000e+00,9.214550e+02
209,Nichtwohnen,Energie: Endenergieträger,Endenergie,Biomasse,GWh/a,6.886499e+03,3.481268e+03,7.128664e+03
210,Nichtwohnen,Energie: Endenergieträger,Endenergie,Heizöl,GWh/a,1.056335e+04,0.000000e+00,6.125292e+02
211,Nichtwohnen,Energie: Endenergieträger,Endenergie,Sonstige thermisch,GWh/a,0.000000e+00,0.000000e+00,0.000000e+00


c:\Users\ma0026.NB-MA0026\anaconda3\Lib\site-packages\kaleido\_sync_server.py:11: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.

You can however, use the Kaleido API directly which will work with your plotly version. `kaleido.write_fig(...)`, for example. Please see the kaleido documentation.




## Vergleichbarkeit je Kennwert

- **Was:** Status je Kennwert über alle Szenarien (schlechtester Status gegenüber der Referenz).
- **Eingaben:** Kennwerttabelle
- **Rechnet oder liest:** rechnet Status
- **Annahmen:** siehe oben
- **Interpretation:** Zeilen mit NICHT_VERGLEICHBAR oder KENNWERT_FEHLT werden ohne Differenzen ausgewiesen.
- **Sperrende Fehler:** –

In [8]:
v = erg['vergleichbarkeit']
display(v[v.Status != sv.VERGLEICHBAR])

,Bereich,Gruppe,Kennwert,Kategorie,Einheit,Status_Transition_2040_ambitioniert,Status_Transition_2040_Wachstum,Status,Hinweis


## Referenzvergleich

- **Was:** Jedes Szenario gegen die Referenz: absolute Differenz, relative Änderung, Prozentpunkte, mittlere Nettoänderung pro Jahr.
- **Eingaben:** Kennwerttabelle, Szenariojahre
- **Rechnet oder liest:** rechnet Differenzen
- **Annahmen:** Relative Änderung nur bei Referenzwert ≠ 0; Prozentpunkte nur für Anteile (% NGF); Jahresrate nur bei positiver Zeitspanne.
- **Interpretation:** Jahreswerte sind mittlere Nettoänderungen zwischen Zuständen, keine Sanierungs-, Neubau- oder Abrissraten. Gleichjährige Alternativszenarien erhalten keine Jahresrate.
- **Sperrende Fehler:** –

In [9]:
r = erg['referenzvergleich']
display(r[r.Gruppe.isin(['Fläche', 'Qualität'])])
sv.grafik_referenz(erg).show()
fig_energie = sv.grafik_energie(erg)
if fig_energie is not None:
    fig_energie.show()
else:
    print('Keine vergleichbaren Energiekennwerte.')

,Bereich,Gruppe,Kennwert,Kategorie,Einheit,Referenz,Szenario_ID,Jahre,Wert_Referenz,Wert_Szenario,Delta,Relative_Aenderung_pct,Delta_pp,Netto_pro_Jahr,Status,Hinweis
0,Gesamt,Fläche,NGF,,m²,Bestand_2024,Transition_2040_ambitioniert,16,8.898726e+08,9.500000e+08,6.012744e+07,6.756860,NaN,3.757965e+06,VERGLEICHBAR,
1,Gesamt,Fläche,NGF,,m²,Bestand_2024,Transition_2040_Wachstum,16,8.898726e+08,9.500000e+08,6.012744e+07,6.756860,NaN,3.757965e+06,VERGLEICHBAR,
2,Gesamt,Fläche,BGF,,m²,Bestand_2024,Transition_2040_ambitioniert,16,1.112341e+09,1.187500e+09,7.515930e+07,6.756860,NaN,4.697456e+06,VERGLEICHBAR,
3,Gesamt,Fläche,BGF,,m²,Bestand_2024,Transition_2040_Wachstum,16,1.112341e+09,1.187500e+09,7.515930e+07,6.756860,NaN,4.697456e+06,VERGLEICHBAR,
6,Haushalte,Fläche,NGF,,m²,Bestand_2024,Transition_2040_ambitioniert,16,5.656768e+08,6.038988e+08,3.822199e+07,6.756860,NaN,2.388874e+06,VERGLEICHBAR,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
127,Gesamt,Qualität,NGF,Neu_30cm,m²,Bestand_2024,Transition_2040_Wachstum,16,1.235143e+08,3.325000e+08,2.089857e+08,169.199575,NaN,1.306161e+07,VERGLEICHBAR,
128,Gesamt,Qualität,NGF-Anteil,Neu_30cm,% NGF,Bestand_2024,Transition_2040_ambitioniert,16,1.388000e+01,4.462153e+01,3.074153e+01,221.480784,30.741533,1.921346e+00,VERGLEICHBAR,
129,Gesamt,Qualität,NGF-Anteil,Neu_30cm,% NGF,Bestand_2024,Transition_2040_Wachstum,16,1.388000e+01,3.500000e+01,2.112000e+01,152.161383,21.120000,1.320000e+00,VERGLEICHBAR,
130,Gesamt,Qualität,Positiv belegte Typen,Neu_30cm,Anzahl,Bestand_2024,Transition_2040_ambitioniert,16,2.088000e+03,1.680000e+03,-4.080000e+02,-19.540230,NaN,-2.550000e+01,VERGLEICHBAR,


## Paarvergleich

- **Was:** Detailvergleich zweier Zustände: Kategorien je Merkmal, Flächen- und Struktureffekt, minimale Umverteilung, Qualität × Versorgung.
- **Eingaben:** Verteilung.csv der beiden Szenarien
- **Rechnet oder liest:** rechnet Differenzen aus Zustandsflächen
- **Annahmen:** Zerlegung Δ = Flächeneffekt + Struktureffekt (symmetrisch). Minimale Umverteilung nur bei gleicher Gesamt-NGF.
- **Interpretation:** Die minimale Umverteilung ist eine Untergrenze, keine Flussrekonstruktion.
- **Sperrende Fehler:** Kontrollsummen verletzt

In [10]:
p = erg['paarvergleich']
if p is None:
    print('Kein Paar gewählt.')
else:
    print(p['szenario_a'], '→', p['szenario_b'], '|', p['jahre'], 'Jahre | gleiche NGF:', p['gleiche_ngf'])
    display(p['kontrollen'])
    display(p['untergrenzen'])
    display(p['merkmale'])
    sv.grafik_paar_matrix(p).show()

Bestand_2024 → Transition_2040_ambitioniert | 16 Jahre | gleiche NGF: False


,Pruefung,Ist,Soll,Status
0,Delta-Summe geometry,60127442.0,60127442.0,OK
1,Delta-Summe quality,60127442.0,60127442.0,OK
2,Delta-Summe energy,60127442.0,60127442.0,OK
3,Delta-Summe pv,60127442.0,60127442.0,OK
4,Delta-Summe flex,60127442.0,60127442.0,OK
5,Delta-Summe weather,60127442.0,60127442.0,OK
6,Delta-Summe mass,60127442.0,60127442.0,OK
7,Typen-Delta-Summe,60127442.0,60127442.0,OK
8,Qualität×Versorgung-Delta-Summe,60127442.0,60127442.0,OK


,Merkmal,Minimale_Umverteilung_m2,Anteil_am_Bestand_pct
0,Geometrie,NaN,NaN
1,Qualität,NaN,NaN
2,Planversorgung,NaN,NaN
3,PV,NaN,NaN
4,Flexibilität,NaN,NaN
5,Wetter,NaN,NaN
6,Speichermasse,NaN,NaN
7,Vollständiger Typenschlüssel,NaN,NaN


,Merkmal,Eigenschaft,Kategorie,NGF_A_m2,NGF_B_m2,Delta_NGF_m2,Anteil_A_pct,Anteil_B_pct,Delta_pp,Relative_Aenderung_pct,Netto_m2_pro_Jahr,Netto_pp_pro_Jahr,Flaecheneffekt_m2,Struktureffekt_m2
0,Geometrie,geometry,EFH_1G,4.812062e+07,5.137206e+07,3.251443e+06,5.407586,5.407586,0.000000e+00,6.756860e+00,2.032152e+05,0.000000e+00,3.251443e+06,0.000000e+00
1,Geometrie,geometry,GWB_7G,6.868751e+07,7.332863e+07,4.641119e+06,7.718803,7.718803,1.387779e-15,6.756860e+00,2.900699e+05,8.673617e-17,4.641119e+06,1.276668e-08
2,Geometrie,geometry,EFH_2G,1.244753e+08,1.328859e+08,8.410623e+06,13.987994,13.987994,0.000000e+00,6.756860e+00,5.256639e+05,0.000000e+00,8.410623e+06,0.000000e+00
3,Geometrie,geometry,NiWo_2G,1.084758e+08,1.158054e+08,7.329559e+06,12.190040,12.190040,-2.775558e-15,6.756860e+00,4.580975e+05,-1.734723e-16,7.329559e+06,-2.553336e-08
4,Geometrie,geometry,NiWo_3G,1.436006e+08,1.533035e+08,9.702892e+06,16.137211,16.137211,8.326673e-15,6.756860e+00,6.064307e+05,5.204170e-16,9.702892e+06,7.660008e-08
5,Geometrie,geometry,RH_2G,1.566610e+08,1.672463e+08,1.058536e+07,17.604875,17.604875,-5.551115e-15,6.756860e+00,6.615851e+05,-3.469447e-16,1.058536e+07,-5.106672e-08
6,Geometrie,geometry,MFH_2G,7.972285e+07,8.510961e+07,5.386761e+06,8.958906,8.958906,-1.387779e-15,6.756860e+00,3.366726e+05,-8.673617e-17,5.386761e+06,-1.276668e-08
7,Geometrie,geometry,MFH_4G,5.484012e+07,5.854559e+07,3.705470e+06,6.162694,6.162694,-1.387779e-15,6.756860e+00,2.315919e+05,-8.673617e-17,3.705470e+06,-1.276668e-08
8,Geometrie,geometry,NiWo_1G,7.211929e+07,7.699229e+07,4.872999e+06,8.104451,8.104451,-1.387779e-15,6.756860e+00,3.045625e+05,-8.673617e-17,4.872999e+06,-1.276668e-08
9,Geometrie,geometry,GWB_5G,3.316947e+07,3.541069e+07,2.241215e+06,3.727441,3.727441,0.000000e+00,6.756860e+00,1.400759e+05,0.000000e+00,2.241215e+06,0.000000e+00


## Export

- **Was:** Speichert den Vergleich geprüft nach `results/vergleiche/<VERGLEICH>/` und liest ihn zur Kontrolle wieder ein.
- **Eingaben:** Vergleichsergebnis
- **Rechnet oder liest:** schreibt nur bei `EXPORTIEREN=True`
- **Annahmen:** Schreiben über Arbeitsordner; vorhandener Ordner nur mit `ERSETZEN=True` ersetzt.
- **Interpretation:** –
- **Sperrende Fehler:** Ordner existiert und `ERSETZEN=False`

In [11]:
if EXPORTIEREN:
    ziel = sv.exportiere(erg, ersetzen=ERSETZEN)
    geladen = sv.lade_vergleich(VERGLEICH, PROJEKT)
    print('Gespeichert:', ziel.relative_to(ROOT).as_posix(), '|', geladen['integritaet'])
else:
    print('EXPORTIEREN=False: nichts gespeichert.')

Gespeichert: results/vergleiche/Bestand_Transition_ambitioniert_2040 | OK (SHA-256 der Ausgaben geprüft)
